In [ ]:
import sys
sys.executable

In [ ]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio

In [ ]:
!{sys.executable} -m pip install matplotlib scikit-learn pillow tqdm

In [1]:
import torch, torchvision
import matplotlib
import sklearn
from PIL import Image
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset

print("All imports successful ✅")


All imports successful ✅


In [4]:
import os, glob
from collections import Counter
#tells me whats inside my dataset


DATA_DIR = r"C:\Users\vicso\Downloads\USAcurrency\Data"


classes = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

counts = {}
for c in classes:
    folder = os.path.join(DATA_DIR, c)
    imgs = (
        glob.glob(os.path.join(folder, "*.jpg")) +
        glob.glob(os.path.join(folder, "*.JPG")) +
        glob.glob(os.path.join(folder, "*.png"))
    )
    counts[c] = len(imgs)

print("Classes:")
for k, v in counts.items():
    print(f"{k}: {v}")

print("\nTotal images:", sum(counts.values()))


Classes:
10 Dollar: 295
100 Dollar: 299
2 Dollar: 302
5 Dollar: 289
50 Dollar: 304

Total images: 1489


In [5]:
from pathlib import Path

BASE_DIR = Path(r"C:\Users\vicso\Downloads\USAcurrency")
TRAIN_DIR = BASE_DIR / "Data"
TEST_DIR   = BASE_DIR / "Test"


In [6]:
import os
from torchvision.datasets import ImageFolder
from PIL import Image

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"}

def is_valid_file(path: str) -> bool:
    return os.path.splitext(path)[1] in IMG_EXTS

# quick check: how many non-images are present?
def count_non_images(root):
    n = 0
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if os.path.splitext(f)[1] not in IMG_EXTS:
                n += 1
    return n

print("Non-images in Data:", count_non_images(TRAIN_DIR))
print("Non-images in Test:", count_non_images(TEST_DIR))


Non-images in Data: 0
Non-images in Test: 0


In [7]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

# --- Grayscale transforms ---
# num_output_channels=3 keeps 3 channels so ResNet50 works without changing its first layer.
train_tfms_gray = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomRotation(180),        # keep your augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_tfms_gray = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# --- Grayscale datasets ---
data_ds_gray = ImageFolder(root=str(TRAIN_DIR), transform=train_tfms_gray, is_valid_file=is_valid_file)
test_ds_gray = ImageFolder(root=str(TEST_DIR),  transform=eval_tfms_gray,  is_valid_file=is_valid_file)

print("Classes:", data_ds_gray.classes)
print("Data size:", len(data_ds_gray))
print("Test size:", len(test_ds_gray))



Classes: ['10 Dollar', '100 Dollar', '2 Dollar', '5 Dollar', '50 Dollar']
Data size: 800
Test size: 200


In [8]:
import numpy as np
from torch.utils.data import Subset

SEED = 42
VAL_FRAC = 0.30
rng = np.random.default_rng(SEED)

targets = np.array(data_ds_gray.targets)
num_classes = len(data_ds_gray.classes)

train_indices, val_indices = [], []

for c in range(num_classes):
    idx_c = np.where(targets == c)[0]
    rng.shuffle(idx_c)

    n_val = int(round(len(idx_c) * VAL_FRAC))
    val_indices.extend(idx_c[:n_val].tolist())
    train_indices.extend(idx_c[n_val:].tolist())

rng.shuffle(train_indices)
rng.shuffle(val_indices)

# IMPORTANT: val should NOT use augmentation, so create an eval-transform view of the same Data folder
data_ds_gray_eval = ImageFolder(root=str(TRAIN_DIR), transform=eval_tfms_gray, is_valid_file=is_valid_file)

train_split_gray = Subset(data_ds_gray,      train_indices)     # augmented grayscale
val_split_gray   = Subset(data_ds_gray_eval, val_indices)       # non-augmented grayscale

print("Gray Train split size:", len(train_split_gray))
print("Gray Val split size:", len(val_split_gray))
print("Gray Test size (separate folder):", len(test_ds_gray))


Gray Train split size: 560
Gray Val split size: 240
Gray Test size (separate folder): 200


In [9]:

train_loader_gray = DataLoader(train_split_gray, batch_size=32, shuffle=True,  num_workers=0)
val_loader_gray   = DataLoader(val_split_gray,   batch_size=32, shuffle=False, num_workers=0)
test_loader_gray  = DataLoader(test_ds_gray,     batch_size=32, shuffle=False, num_workers=0)


In [10]:
import torch
from torch import nn
from torchvision import models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(data_ds_gray.classes))
model = model.to(DEVICE)


Device: cpu


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)


In [12]:
from tqdm import tqdm

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    return total_loss / total, total_correct / total

for epoch in range(1, 6):
    train_loss, train_acc = run_epoch(model, train_loader_gray, train=True)
    val_loss, val_acc = run_epoch(model, val_loader_gray, train=False)
    print(f"Epoch {epoch}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")


Epoch 1: train_acc=0.432, val_acc=0.596


Epoch 2: train_acc=0.793, val_acc=0.771


Epoch 3: train_acc=0.934, val_acc=0.925


Epoch 4: train_acc=0.963, val_acc=0.983


Epoch 5: train_acc=0.986, val_acc=0.983


In [13]:
best_val_acc = -1.0
BEST_PATH = "resnet50_thai_greyscale_best.pth"

for epoch in range(1, 6):
    train_loss, train_acc = run_epoch(model, train_loader_gray, train=True)
    val_loss, val_acc = run_epoch(model, val_loader_gray, train=False)
    print(f"Epoch {epoch}: train_acc={train_acc:.3f}, val_acc={val_acc:.3f}, train_loss ={train_loss:.3f}, val_loss = {val_loss:.3f}" )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_PATH)
        print(f"✅ Saved new best model (val_acc={best_val_acc:.3f})")


Epoch 1: train_acc=0.979, val_acc=0.979, train_loss =0.063, val_loss = 0.055
✅ Saved new best model (val_acc=0.979)


Epoch 2: train_acc=0.993, val_acc=0.979, train_loss =0.034, val_loss = 0.058


Epoch 3: train_acc=1.000, val_acc=0.979, train_loss =0.015, val_loss = 0.066


Epoch 4: train_acc=0.998, val_acc=0.988, train_loss =0.017, val_loss = 0.032
✅ Saved new best model (val_acc=0.988)


Epoch 5: train_acc=0.998, val_acc=0.988, train_loss =0.016, val_loss = 0.035


In [15]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Load best checkpoint
BEST_PATH = "resnet50_thai_greyscale_best.pth"
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for x, y in test_loader_gray:
        x = x.to(DEVICE)
        preds = model(x).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

cm = confusion_matrix(all_true, all_preds)
print("Confusion matrix (TEST):\n", cm)
print("\nReport (TEST):\n", classification_report(all_true, all_preds, target_names=test_ds_gray.classes))


Confusion matrix (TEST):
 [[37  3  0  0  0]
 [ 0 40  0  0  0]
 [ 0  0 40  0  0]
 [ 0  0  0 40  0]
 [ 0  0  0  0 40]]

Report (TEST):
               precision    recall  f1-score   support

   10 Dollar       1.00      0.93      0.96        40
  100 Dollar       0.93      1.00      0.96        40
    2 Dollar       1.00      1.00      1.00        40
    5 Dollar       1.00      1.00      1.00        40
   50 Dollar       1.00      1.00      1.00        40

    accuracy                           0.98       200
   macro avg       0.99      0.98      0.98       200
weighted avg       0.99      0.98      0.98       200

